# 05 Uncertainty experiment (Experiment C)

**Question.** Do predicted variances track error, and do split-conformal intervals reach their nominal coverage on an independent partition?

**Type.** Run/inspection ([spec](../docs/experiments/c-uncertainty-aware.md)). Calibration is fitted on the calibration partition only.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
from ct_iqa.config.loader import load_study_config
from ct_iqa.experiment.runs import latest_completed_run, read_predictions
CONFIG = "configs/experiments/c_uncertainty_aware.yaml"
cfg = load_study_config(CONFIG)
print(cfg.run.experiment_id, cfg.model.name, cfg.loss.name, "bn_eps =", cfg.model.bn_eps)

## Run (explicit, expensive)
Set `RUN_TRAINING = True` only with an approved compute budget; this trains for the configured number of epochs and writes a new immutable run directory.

In [ ]:
RUN_TRAINING = False
DEVICE = "cpu"   # use "cuda" when available
if RUN_TRAINING:
    import dataclasses
    from ct_iqa.experiment.run import run_experiment
    cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, device=DEVICE))
    run_dir = run_experiment(cfg, config_path=CONFIG)
else:
    run_dir = latest_completed_run(cfg.run.experiment_id)
print("run directory:", run_dir if run_dir else "NO COMPLETED RUN FOUND (status PENDING)")

In [ ]:
if run_dir:
    print(json.loads((run_dir / "metrics_validation.json").read_text()))
    hist = json.loads((run_dir / "history.json").read_text())
    print("selected epoch (1-based):", hist["best_epoch"], "policy:", hist["selection_policy"])

In [ ]:
if run_dir:
    cal = json.loads((run_dir / "calibration.json").read_text())
    print({k: cal[k] for k in ("target_coverage", "n_calibration", "rank_k", "quantile_q", "unbounded")})
    print("unclipped:", cal["unclipped"]); print("clipped to [0,4]:", cal["clipped_to_0_4"])
    print("NLL (0-4 scale):", cal["nll_score_scale"], "| Spearman(sigma, |err|):", cal["sigma_error_spearman"])
    print("coverage by sigma tercile:", cal["coverage_by_sigma_tercile"])
    print("Reminder: nominal coverage is not clinical reliability; aleatoric only; exchangeability unverified.")